# PSD Notebook Model Comparison

This notebook compares the PSD diffusion models trained from the notebooks in `SpecDiff`.

It auto-discovers saved `.pth` files under `models/` and `checkpoints/`, rebuilds the matching backbone, and evaluates each model in two inference modes:

- pure-noise generation
- restoration from a noisy prior

It is intended for the PSD notebook models created in this repo, including the `Flex`, weighted/restoration `x`, and SDE notebook variants.


In [ ]:
from pathlib import Path
import os
import sys
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm


def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / 'SpecDiff',
        Path('/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/SpecDiff'),
    ]
    for candidate in candidates:
        if (candidate / 'Backbone.py').exists() and (candidate / 'ParamDiffuser.py').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find the SpecDiff repo root.')


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import Backbone
import SDEBackbone
import Transformer
import ParamDiffuser as diff

try:
    from skimage.metrics import structural_similarity as skimage_ssim
    HAS_SKIMAGE = True
except Exception:
    skimage_ssim = None
    HAS_SKIMAGE = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEFAULT_PSD_ROOT = Path('/Users/27171653/Desktop/PhD/Highlight-modelling/PSD_Dataset/PSD_Dataset')
PSD_ROOT = Path(os.environ.get('PSD_DATASET_ROOT', str(DEFAULT_PSD_ROOT))).expanduser()

TEST_SPLIT = 'PSD_Test'
SEARCH_DIRS = [ROOT / 'models', ROOT / 'checkpoints']
MODEL_NAME_FILTER = None  # regex string or None
INCLUDE_CHECKPOINT_MODELS = True
PREFER_EMA = True

EVAL_MODES = ('pure_noise', 'restoration')
MAX_EVAL_SAMPLES = None
FIXED_CASE_INDEX = 0
NOISE_SEED = 123

RESTORE_START_STEP_OVERRIDE = None
RESTORE_DETERMINISTIC = True
RESTORE_CLIP_X0 = True
RESTORATION_PRIOR = 'auto'  # 'auto', 'glossy', or 'zero'
SHOW_PROGRESS = True
MAX_VISUAL_MODELS = None

print(f'repo root: {ROOT}')
print(f'device: {DEVICE}')
print(f'PSD root: {PSD_ROOT}')
print(f'search dirs: {SEARCH_DIRS}')
print(f'SSIM available: {HAS_SKIMAGE}')


In [ ]:
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
PIL_BILINEAR = getattr(Image, 'Resampling', Image).BILINEAR


def normalize_image_key(name: str) -> str:
    stem = Path(name).stem.lower()
    stem = stem.replace('specular', '').replace('glossy', '').replace('diffuse', '')
    return re.sub(r'[^a-z0-9]+', '', stem)


def pair_image_paths(glossy_dir: Path, diffuse_dir: Path):
    glossy_candidates = [path for path in glossy_dir.iterdir() if path.suffix.lower() in VALID_EXTS]
    diffuse_candidates = [path for path in diffuse_dir.iterdir() if path.suffix.lower() in VALID_EXTS]

    glossy_files = {path.name: path for path in glossy_candidates}
    diffuse_files = {path.name: path for path in diffuse_candidates}
    exact_names = sorted(set(glossy_files) & set(diffuse_files))
    if exact_names:
        return [(glossy_files[name], diffuse_files[name], name) for name in exact_names]

    glossy_by_key = {}
    for path in glossy_candidates:
        key = normalize_image_key(path.name)
        if key in glossy_by_key:
            raise RuntimeError(f'Duplicate glossy normalized key {key!r} in {glossy_dir}')
        glossy_by_key[key] = path

    diffuse_by_key = {}
    for path in diffuse_candidates:
        key = normalize_image_key(path.name)
        if key in diffuse_by_key:
            raise RuntimeError(f'Duplicate diffuse normalized key {key!r} in {diffuse_dir}')
        diffuse_by_key[key] = path

    common_keys = sorted(set(glossy_by_key) & set(diffuse_by_key))
    if not common_keys:
        raise RuntimeError(f'No paired PSD samples found in {glossy_dir} and {diffuse_dir}')

    return [(glossy_by_key[key], diffuse_by_key[key], glossy_by_key[key].name) for key in common_keys]


def get_split_dirs(psd_root: Path, split_name: str) -> tuple[Path, Path]:
    split_dir = psd_root / split_name
    glossy_dir = split_dir / f'{split_name}_specular'
    diffuse_dir = split_dir / f'{split_name}_diffuse'
    if not glossy_dir.exists():
        raise FileNotFoundError(f'Missing glossy directory: {glossy_dir}')
    if not diffuse_dir.exists():
        raise FileNotFoundError(f'Missing diffuse directory: {diffuse_dir}')
    return glossy_dir, diffuse_dir


def load_rgb_tensor(path: Path, image_size: int) -> torch.Tensor:
    image = Image.open(path).convert('RGB')
    image = image.resize((image_size, image_size), resample=PIL_BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).contiguous()


def rgb_to_luminance(image: torch.Tensor) -> torch.Tensor:
    weights = image.new_tensor([0.2990, 0.5870, 0.1140]).view(3, 1, 1)
    return (image * weights).sum(dim=0, keepdim=True)


def conv2d_single_channel(image: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    return F.conv2d(image.unsqueeze(0), kernel.to(image.device, image.dtype), padding=1).squeeze(0)


def make_edge_condition(glossy: torch.Tensor) -> torch.Tensor:
    gray = rgb_to_luminance(glossy)
    clip_value = torch.quantile(gray.flatten(), 0.98).clamp_min(1e-6)
    gray = gray.clamp(max=clip_value) / clip_value

    blur_kernel = torch.tensor(
        [[1.0, 2.0, 1.0], [2.0, 4.0, 2.0], [1.0, 2.0, 1.0]],
        dtype=torch.float32,
    ).view(1, 1, 3, 3) / 16.0
    gray = conv2d_single_channel(gray, blur_kernel)

    sobel_x = torch.tensor(
        [[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]],
        dtype=torch.float32,
    ).view(1, 1, 3, 3)
    sobel_y = torch.tensor(
        [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]],
        dtype=torch.float32,
    ).view(1, 1, 3, 3)

    grad_x = conv2d_single_channel(gray, sobel_x)
    grad_y = conv2d_single_channel(gray, sobel_y)
    edges = torch.sqrt(grad_x.pow(2) + grad_y.pow(2) + 1e-12)
    return edges / edges.amax().clamp_min(1e-6)


def make_condition(glossy: torch.Tensor, condition_mode: str) -> torch.Tensor:
    if condition_mode == 'glossy_rgb':
        return glossy
    if condition_mode == 'edge_1ch':
        return make_edge_condition(glossy)
    if condition_mode == 'none':
        return torch.zeros((1, glossy.shape[1], glossy.shape[2]), dtype=glossy.dtype)
    raise ValueError(f'Unknown condition mode: {condition_mode}')


def build_case_tensors(glossy_path: Path, diffuse_path: Path, image_size: int, condition_mode: str, device: torch.device):
    glossy = load_rgb_tensor(glossy_path, image_size)
    diffuse = load_rgb_tensor(diffuse_path, image_size)
    condition = make_condition(glossy, condition_mode)
    return (
        glossy.unsqueeze(0).to(device),
        condition.unsqueeze(0).to(device),
        diffuse.unsqueeze(0).to(device),
    )


def make_case_noise(shape, seed: int, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    noise = torch.randn(shape, generator=generator, dtype=dtype)
    return noise.to(device)


In [ ]:
class ConditionedSDEUNet(SDEBackbone.UNetWithTransformer):
    def __init__(self, noise_steps: int = 1000, time_dim: int = 256, size: int = 32, depth: int = 4, conditioning_channels: int = 1):
        super().__init__(noise_steps=noise_steps, time_dim=time_dim, size=size, depth=depth)
        self.conditioning_channels = conditioning_channels
        self.dit = Transformer.Transformer_S_2(
            input_size=self.image_size // (2 ** self.depth),
            in_channels=self.dit_channels,
            conditioning_channels=conditioning_channels,
            learn_sigma=False,
        )
        self.dit_proj_in = nn.Conv2d(self.dit_channels, self.dit.in_channels, kernel_size=1)
        self.dit_proj_out = nn.Conv2d(self.dit.out_channels, self.dit_channels, kernel_size=1)


def load_checkpoint(path: Path):
    return torch.load(path, map_location='cpu')


def extract_state_dict(checkpoint: dict, prefer_ema: bool = True):
    if isinstance(checkpoint, dict):
        if prefer_ema and 'ema_model_state_dict' in checkpoint:
            return checkpoint['ema_model_state_dict'], 'ema_model_state_dict'
        if 'model_state_dict' in checkpoint:
            return checkpoint['model_state_dict'], 'model_state_dict'
    return checkpoint, 'state_dict'


def infer_image_size(path: Path, checkpoint: dict) -> int:
    text = f'{path.as_posix()} {path.parent.as_posix()}'
    match = re.search(r'Flex(\d+)', text)
    if match:
        return int(match.group(1))
    for part in reversed(path.parts):
        if part.isdigit():
            size = int(part)
            if size in {16, 32, 64, 128, 256, 512}:
                return size
    return int(checkpoint.get('image_size', 32))


def infer_train_type(path: Path, checkpoint: dict) -> str:
    if isinstance(checkpoint, dict) and checkpoint.get('train_type') is not None:
        return str(checkpoint['train_type'])
    match = re.search(r'PSD_Difference_([exv])_Flex', path.name)
    if match:
        return match.group(1)
    lowered = path.as_posix().lower()
    if 'weightedx' in lowered or 'restorationx' in lowered or 'diffuseonly' in lowered:
        return 'x'
    return 'e'


def infer_model_info(path: Path) -> dict:
    checkpoint = load_checkpoint(path)
    state_dict, state_key = extract_state_dict(checkpoint, prefer_ema=PREFER_EMA)
    rel_path = path.relative_to(ROOT) if path.is_relative_to(ROOT) else path
    lowered = rel_path.as_posix().lower()

    if (isinstance(checkpoint, dict) and checkpoint.get('conditioning') == 'none') or 'uncondattention' in lowered:
        family = 'sde_uncond_attention'
    elif (isinstance(checkpoint, dict) and checkpoint.get('condition_mode') is not None) or 'sde_diffuseonly_' in lowered:
        family = 'sde_transformer'
    else:
        family = 'flex'

    if family.startswith('sde'):
        target_mode = 'diffuse'
    else:
        target_mode = checkpoint.get('target_mode') if isinstance(checkpoint, dict) else None
        if target_mode is None:
            target_mode = 'difference' if 'difference' in lowered else 'diffuse'

    if family == 'sde_uncond_attention':
        condition_mode = 'none'
    elif family == 'sde_transformer':
        condition_mode = checkpoint.get('condition_mode') if isinstance(checkpoint, dict) else None
        if condition_mode is None:
            condition_mode = 'edge_1ch' if 'edge_1ch' in lowered else 'glossy_rgb'
    else:
        condition_mode = 'glossy_rgb'

    condition_channels = checkpoint.get('condition_channels') if isinstance(checkpoint, dict) else None
    if condition_channels is None:
        condition_channels = 1 if condition_mode in {'edge_1ch', 'none'} else 3

    image_size = infer_image_size(path, checkpoint if isinstance(checkpoint, dict) else {})
    noise_steps = int(checkpoint.get('noise_steps', 200)) if isinstance(checkpoint, dict) else 200
    depth = int(checkpoint.get('depth', 4 if family.startswith('sde') else 2)) if isinstance(checkpoint, dict) else (4 if family.startswith('sde') else 2)
    train_type = infer_train_type(path, checkpoint if isinstance(checkpoint, dict) else {})
    restore_start_step = checkpoint.get('restore_start_step') if isinstance(checkpoint, dict) else None
    has_ema = isinstance(checkpoint, dict) and 'ema_model_state_dict' in checkpoint

    if path.name == 'model.pth':
        display_name = rel_path.parent.as_posix() + '/model.pth'
    else:
        display_name = rel_path.as_posix()

    return {
        'path': path,
        'display_name': display_name,
        'family': family,
        'train_type': train_type,
        'target_mode': target_mode,
        'condition_mode': condition_mode,
        'condition_channels': int(condition_channels),
        'image_size': image_size,
        'noise_steps': noise_steps,
        'depth': depth,
        'restore_start_step': None if restore_start_step is None else int(restore_start_step),
        'has_ema': has_ema,
        'state_key': state_key,
        'checkpoint_keys': sorted(checkpoint.keys()) if isinstance(checkpoint, dict) else [],
    }


def discover_models(search_dirs, include_checkpoint_models: bool = True, name_filter: str | None = None):
    discovered = []
    regex = re.compile(name_filter) if name_filter else None
    for search_dir in search_dirs:
        if not search_dir.exists():
            continue
        for path in sorted(search_dir.rglob('*.pth')):
            lowered = path.as_posix().lower()
            if path.name == 'model.pth' and not include_checkpoint_models:
                continue
            if 'psd' not in lowered and 'sde' not in lowered:
                continue
            if regex and not regex.search(path.as_posix()):
                continue
            try:
                discovered.append(infer_model_info(path))
            except Exception as exc:
                discovered.append(
                    {
                        'path': path,
                        'display_name': str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path),
                        'family': 'unparsed',
                        'train_type': 'unknown',
                        'target_mode': 'unknown',
                        'condition_mode': 'unknown',
                        'condition_channels': np.nan,
                        'image_size': np.nan,
                        'noise_steps': np.nan,
                        'depth': np.nan,
                        'restore_start_step': np.nan,
                        'has_ema': False,
                        'state_key': 'unparsed',
                        'checkpoint_keys': [],
                        'error': repr(exc),
                    }
                )
    return discovered


In [ ]:
def build_model_from_info(info: dict, device: torch.device, prefer_ema: bool = True):
    checkpoint = load_checkpoint(info['path'])
    state_dict, state_key = extract_state_dict(checkpoint, prefer_ema=prefer_ema)

    if info['family'] == 'flex':
        model = Backbone.Flex(size=int(info['image_size']), noise_steps=int(info['noise_steps']))
    elif info['family'] == 'sde_transformer':
        model = ConditionedSDEUNet(
            noise_steps=int(info['noise_steps']),
            size=int(info['image_size']),
            depth=int(info['depth']),
            conditioning_channels=int(info['condition_channels']),
        )
    elif info['family'] == 'sde_uncond_attention':
        model = SDEBackbone.UNetWithAttention(noise_steps=int(info['noise_steps']), depth=int(info['depth']))
    else:
        raise ValueError(f"Unsupported family: {info['family']}")

    model.load_state_dict(state_dict, strict=True)
    model = model.to(device)
    model.eval()
    diffuser = diff.CosSchDiffuser(steps=int(info['noise_steps']), device=device)
    return model, diffuser, state_key


def prediction_to_diffuse(prediction: torch.Tensor, glossy: torch.Tensor, target_mode: str) -> torch.Tensor:
    if target_mode == 'difference':
        return glossy - prediction
    return prediction


def target_space_limits(target_mode: str) -> tuple[float, float]:
    if target_mode == 'difference':
        return -1.0, 1.0
    return 0.0, 1.0


def build_restoration_prior(glossy: torch.Tensor, target_mode: str, prior_mode: str = 'auto') -> torch.Tensor:
    if prior_mode == 'glossy':
        return glossy.clamp(0.0, 1.0)
    if prior_mode == 'zero':
        return torch.zeros_like(glossy)
    if target_mode == 'difference':
        return torch.zeros_like(glossy)
    return glossy.clamp(0.0, 1.0)


def sample_pure_noise(model: torch.nn.Module, diffuser: diff.Diffuser, condition: torch.Tensor, train_type: str, initial_noise: torch.Tensor) -> torch.Tensor:
    return diffuser.sample_from_noise(
        model,
        condition,
        parameterization=train_type,
        show_progress=False,
        initial_noise=initial_noise,
    )


def sample_restoration(
    model: torch.nn.Module,
    diffuser: diff.Diffuser,
    glossy: torch.Tensor,
    condition: torch.Tensor,
    train_type: str,
    target_mode: str,
    start_step: int,
    deterministic: bool,
    initial_noise: torch.Tensor,
    clip_x0: bool,
    prior_mode: str = 'auto',
) -> torch.Tensor:
    if not 0 <= start_step < diffuser.steps:
        raise ValueError(f'start_step must be in [0, {diffuser.steps - 1}], got {start_step}')

    prior = build_restoration_prior(glossy, target_mode, prior_mode=prior_mode)
    noise = initial_noise.to(glossy.device, dtype=glossy.dtype)
    t0 = torch.full((glossy.shape[0],), start_step, dtype=torch.long, device=glossy.device)
    x_t = diffuser.forward_diffusion(prior, t0, noise)

    with torch.no_grad():
        for step in range(start_step, -1, -1):
            t = torch.full((glossy.shape[0],), step, dtype=torch.long, device=glossy.device)
            model_output = model(x_t, t, condition)
            pred_x0, _ = diffuser.predict_x0_and_noise(x_t, t, model_output, parameterization=train_type)

            if clip_x0:
                low, high = target_space_limits(target_mode)
                pred_x0 = pred_x0.clamp(low, high)

            if step == 0:
                return pred_x0

            mean, variance = diffuser.q_posterior(pred_x0, x_t, t)
            if deterministic:
                x_t = mean
            else:
                x_t = mean + torch.sqrt(variance.clamp_min(1e-20)) * torch.randn_like(x_t)

    return x_t


def compute_ssim(pred_rgb: torch.Tensor, target_rgb: torch.Tensor) -> float:
    if not HAS_SKIMAGE:
        return float('nan')
    pred_np = pred_rgb.detach().cpu().permute(1, 2, 0).numpy().clip(0.0, 1.0)
    target_np = target_rgb.detach().cpu().permute(1, 2, 0).numpy().clip(0.0, 1.0)
    return float(skimage_ssim(target_np, pred_np, data_range=1.0, channel_axis=2))


def compute_metrics(pred_target: torch.Tensor, true_target: torch.Tensor, pred_diffuse: torch.Tensor, true_diffuse: torch.Tensor) -> dict:
    target_mse = float(F.mse_loss(pred_target, true_target).item())
    target_mae = float((pred_target - true_target).abs().mean().item())

    diffuse_mse = float(F.mse_loss(pred_diffuse, true_diffuse).item())
    diffuse_mae = float((pred_diffuse - true_diffuse).abs().mean().item())
    pred_diffuse_clipped = pred_diffuse.clamp(0.0, 1.0)
    diffuse_psnr = float((10.0 * torch.log10(1.0 / F.mse_loss(pred_diffuse_clipped, true_diffuse).clamp_min(1e-10))).item())
    diffuse_ssim = compute_ssim(pred_diffuse_clipped.squeeze(0), true_diffuse.squeeze(0))

    return {
        'target_mse': target_mse,
        'target_mae': target_mae,
        'diffuse_mse': diffuse_mse,
        'diffuse_mae': diffuse_mae,
        'diffuse_psnr': diffuse_psnr,
        'diffuse_ssim': diffuse_ssim,
    }


In [ ]:
discovered_models = discover_models(
    SEARCH_DIRS,
    include_checkpoint_models=INCLUDE_CHECKPOINT_MODELS,
    name_filter=MODEL_NAME_FILTER,
)

models_df = pd.DataFrame(discovered_models)
if models_df.empty:
    print('No notebook-trained PSD diffusion models found in the configured search dirs.')
    selected_models = []
else:
    display_columns = [
        'display_name',
        'family',
        'train_type',
        'target_mode',
        'condition_mode',
        'condition_channels',
        'image_size',
        'noise_steps',
        'depth',
        'restore_start_step',
        'has_ema',
        'state_key',
    ]
    if 'error' in models_df.columns:
        display_columns.append('error')
    display(models_df[display_columns].sort_values('display_name').reset_index(drop=True))
    selected_models = [row for row in discovered_models if row.get('family') != 'unparsed']
    print(f'selected models: {len(selected_models)}')


In [ ]:
glossy_dir, diffuse_dir = get_split_dirs(PSD_ROOT, TEST_SPLIT)
test_pairs = pair_image_paths(glossy_dir, diffuse_dir)
if MAX_EVAL_SAMPLES is not None:
    test_pairs = test_pairs[:int(MAX_EVAL_SAMPLES)]
if not test_pairs:
    raise RuntimeError('No PSD test pairs found.')
if not 0 <= FIXED_CASE_INDEX < len(test_pairs):
    raise IndexError(f'FIXED_CASE_INDEX {FIXED_CASE_INDEX} is out of range for {len(test_pairs)} paired samples.')

print(f'eval split: {TEST_SPLIT}')
print(f'paired cases: {len(test_pairs)}')
print(f'fixed case: {test_pairs[FIXED_CASE_INDEX][2]}')

results = []
visual_records = []

model_iterator = selected_models
if SHOW_PROGRESS:
    model_iterator = tqdm(selected_models, desc='Models', dynamic_ncols=True)

for model_info in model_iterator:
    try:
        model, diffuser, used_state_key = build_model_from_info(model_info, DEVICE, prefer_ema=PREFER_EMA)
    except Exception as exc:
        for mode in EVAL_MODES:
            results.append(
                {
                    'display_name': model_info['display_name'],
                    'mode': mode,
                    'status': 'load_failed',
                    'error': repr(exc),
                }
            )
        continue

    for mode in EVAL_MODES:
        metric_accumulator = []
        case_iterator = list(enumerate(test_pairs))
        if SHOW_PROGRESS:
            case_iterator = tqdm(case_iterator, desc=f"{model_info['display_name']} | {mode}", leave=False, dynamic_ncols=True)

        for case_idx, (glossy_path, diffuse_path, case_name) in case_iterator:
            glossy, condition, diffuse = build_case_tensors(
                glossy_path,
                diffuse_path,
                image_size=int(model_info['image_size']),
                condition_mode=model_info['condition_mode'],
                device=DEVICE,
            )
            true_target = glossy - diffuse if model_info['target_mode'] == 'difference' else diffuse
            initial_noise = make_case_noise(glossy.shape, NOISE_SEED + case_idx, DEVICE, glossy.dtype)

            with torch.no_grad():
                if mode == 'pure_noise':
                    pred_target = sample_pure_noise(
                        model,
                        diffuser,
                        condition,
                        train_type=model_info['train_type'],
                        initial_noise=initial_noise,
                    )
                elif mode == 'restoration':
                    restore_start_step = model_info['restore_start_step']
                    if RESTORE_START_STEP_OVERRIDE is not None:
                        restore_start_step = int(RESTORE_START_STEP_OVERRIDE)
                    if restore_start_step is None:
                        restore_start_step = min(40, int(model_info['noise_steps']) - 1)
                    pred_target = sample_restoration(
                        model,
                        diffuser,
                        glossy,
                        condition,
                        train_type=model_info['train_type'],
                        target_mode=model_info['target_mode'],
                        start_step=int(restore_start_step),
                        deterministic=RESTORE_DETERMINISTIC,
                        initial_noise=initial_noise,
                        clip_x0=RESTORE_CLIP_X0,
                        prior_mode=RESTORATION_PRIOR,
                    )
                else:
                    raise ValueError(f'Unknown eval mode: {mode}')

            pred_diffuse = prediction_to_diffuse(pred_target, glossy, model_info['target_mode'])
            metrics = compute_metrics(pred_target, true_target, pred_diffuse, diffuse)
            metric_accumulator.append(metrics)

            if case_idx == FIXED_CASE_INDEX:
                visual_records.append(
                    {
                        'display_name': model_info['display_name'],
                        'mode': mode,
                        'glossy': glossy.squeeze(0).detach().cpu(),
                        'diffuse': diffuse.squeeze(0).detach().cpu(),
                        'pred_diffuse': pred_diffuse.squeeze(0).detach().cpu().clamp(0.0, 1.0),
                        'train_type': model_info['train_type'],
                        'target_mode': model_info['target_mode'],
                        'condition_mode': model_info['condition_mode'],
                        'image_size': int(model_info['image_size']),
                        'case_name': case_name,
                    }
                )

        summary = {
            'display_name': model_info['display_name'],
            'mode': mode,
            'family': model_info['family'],
            'train_type': model_info['train_type'],
            'target_mode': model_info['target_mode'],
            'condition_mode': model_info['condition_mode'],
            'image_size': int(model_info['image_size']),
            'noise_steps': int(model_info['noise_steps']),
            'state_key_used': used_state_key,
            'num_cases': len(metric_accumulator),
            'status': 'ok',
        }
        if metric_accumulator:
            for key in metric_accumulator[0].keys():
                summary[key] = float(np.nanmean([item[key] for item in metric_accumulator]))
        results.append(summary)

    del model, diffuser
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
if results_df.empty:
    print('No evaluation results.')
else:
    display_columns = [
        'display_name',
        'mode',
        'family',
        'train_type',
        'target_mode',
        'condition_mode',
        'image_size',
        'state_key_used',
        'num_cases',
        'diffuse_psnr',
        'diffuse_ssim',
        'diffuse_mae',
        'diffuse_mse',
        'target_mae',
        'target_mse',
        'status',
    ]
    display(results_df[display_columns].sort_values(['mode', 'diffuse_psnr'], ascending=[True, False]).reset_index(drop=True))

    ok_df = results_df[results_df['status'] == 'ok'].copy()
    if not ok_df.empty:
        pivot = ok_df.pivot_table(
            index='display_name',
            columns='mode',
            values=['diffuse_psnr', 'diffuse_ssim', 'diffuse_mae', 'diffuse_mse'],
        )
        display(pivot.sort_index())


In [ ]:
if not visual_records:
    print('No visual comparison records available.')
else:
    order = []
    seen = set()
    for record in visual_records:
        if record['display_name'] not in seen:
            seen.add(record['display_name'])
            order.append(record['display_name'])

    if MAX_VISUAL_MODELS is not None:
        order = order[:int(MAX_VISUAL_MODELS)]

    fig, axes = plt.subplots(len(order), 6, figsize=(18, max(3.5 * len(order), 4)), squeeze=False)
    col_titles = ['Glossy input', 'True diffuse', 'Pure-noise diffuse', 'Pure-noise error', 'Restoration diffuse', 'Restoration error']

    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title)

    for row_idx, name in enumerate(order):
        model_records = {record['mode']: record for record in visual_records if record['display_name'] == name}
        reference = next(iter(model_records.values()))
        glossy = reference['glossy']
        diffuse = reference['diffuse']

        axes[row_idx, 0].imshow(glossy.permute(1, 2, 0).numpy().clip(0.0, 1.0))
        axes[row_idx, 1].imshow(diffuse.permute(1, 2, 0).numpy().clip(0.0, 1.0))

        for mode, image_col, error_col in [('pure_noise', 2, 3), ('restoration', 4, 5)]:
            if mode in model_records:
                pred = model_records[mode]['pred_diffuse']
                err = (pred - diffuse).abs().mean(dim=0)
                axes[row_idx, image_col].imshow(pred.permute(1, 2, 0).numpy().clip(0.0, 1.0))
                axes[row_idx, error_col].imshow(err.numpy(), cmap='magma')
            else:
                axes[row_idx, image_col].text(0.5, 0.5, 'n/a', ha='center', va='center')
                axes[row_idx, error_col].text(0.5, 0.5, 'n/a', ha='center', va='center')

        label = (
            f"{name}\n"
            f"{reference['train_type']} | {reference['target_mode']} | {reference['condition_mode']} | {reference['image_size']}px"
        )
        axes[row_idx, 0].set_ylabel(label, rotation=0, labelpad=90, va='center')

    for ax in axes.ravel():
        ax.axis('off')

    plt.tight_layout()
    plt.show()
